In [2]:
import pickle
import pandas as pd
import numpy as np

In [3]:
blind = {33, 35, 36, 38, 39, 41, 42, 43, 53}
ctrlA = {3, 4, 5, 6, 7, 8, 9, 10, 11, 27}
ctrlAV = {12, 13, 14, 15, 16, 17, 18, 19, 22, 32}

participant_to_group = {}

for p in blind:
    participant_to_group[p] = "blind"
for p in ctrlA:
    participant_to_group[p] = "ctrlA"
for p in ctrlAV:
    participant_to_group[p] = "ctrlAV"

run_start_indices = [0, 267, 492, 812, 1137, 1373]

In [4]:
def convert_nested_dict_to_df(data_dict, participant_to_group):
    df_list = []
    
    for participant, roi_dict in data_dict.items():
        for roi, model_dict in roi_dict.items():

            temp = (
                pd.DataFrame(model_dict)
                .reset_index()
                .melt(id_vars="index", var_name="model", value_name="value")
            )

            temp["run"] = np.searchsorted(run_start_indices, temp["index"], side="right")

            temp["participant"] = participant
            temp["roi"] = roi
            temp["roi_type"] = temp["roi"].apply(lambda x: "language" if isinstance(x, (int, float)) else "visual")
            temp["participant_group"] = participant_to_group[participant]

            df_list.append(temp)
    return pd.concat(df_list, ignore_index=True)

In [5]:
with open("january/qwen-text/results.pkl", "rb") as f:
    data = pickle.load(f)

df1 = convert_nested_dict_to_df(data, participant_to_group)

with open("january/qwen-omni/results.pkl", "rb") as f:
    data = pickle.load(f)

df2 = convert_nested_dict_to_df(data, participant_to_group)

with open("january/other/results.pkl", "rb") as f:
    data = pickle.load(f)

df3 = convert_nested_dict_to_df(data, participant_to_group)

In [6]:
df = pd.concat([df1, df2, df3], ignore_index=True)

In [7]:
df

,index,model,value,run,participant,roi,roi_type,participant_group
0,0,qwen-text_layers28-32_conv,0.027176,1,33,2,language,blind
1,1,qwen-text_layers28-32_conv,0.015785,1,33,2,language,blind
2,2,qwen-text_layers28-32_conv,0.030376,1,33,2,language,blind
3,3,qwen-text_layers28-32_conv,0.023361,1,33,2,language,blind
4,4,qwen-text_layers28-32_conv,0.095281,1,33,2,language,blind
...,...,...,...,...,...,...,...,...
5945285,1572,word2vec,-0.071355,6,32,visual,visual,ctrlAV
5945286,1573,word2vec,0.080193,6,32,visual,visual,ctrlAV
5945287,1574,word2vec,0.001791,6,32,visual,visual,ctrlAV
5945288,1575,word2vec,0.013398,6,32,visual,visual,ctrlAV


In [27]:
with open("january/semantic/results_semantic.pkl", "rb") as f:
    semanticresults = pickle.load(f)

print(semanticresults)

{('binder_concreteness', 'Llama_layers_layers7-11'): array([-3.13350466e-02, -3.71863540e-05,  4.27028726e-02, ...,
        1.35689352e-02,  1.35800894e-02, -3.52469296e-02], shape=(1577,)), ('binder_concreteness', 'qwen-omni_audiovideotext_layers12-16_conv'): array([0.05581155, 0.03204107, 0.00559593, ..., 0.07999877, 0.07452918,
       0.05057425], shape=(1577,)), ('binder_concreteness', 'qwen-text_layers28-32_conv'): array([ 0.27001908,  0.0684878 , -0.05568621, ...,  0.22380827,
        0.22496432,  0.01897213], shape=(1577,)), ('binder_concreteness', 'qwen-omni_audiovideotext_layers8-12_conv'): array([ 0.06133621,  0.02462208, -0.00698235, ...,  0.12924191,
        0.128101  ,  0.06967971], shape=(1577,)), ('binder_concreteness', 'qwen-text_layers32-36_conv'): array([ 0.06119269, -0.01965075, -0.00501981, ...,  0.06721024,
        0.06460134,  0.03899974], shape=(1577,)), ('binder_concreteness', 'qwen-text_layers12-16_conv'): array([ 0.13310298,  0.05348171, -0.00835268, ...,  0.1

In [32]:
semantic_models = set(m for _, m in semanticresults.keys())
df_models = set(df["model"].unique())

print("Semantic-only models:", semantic_models - df_models)
print("DF-only models:", df_models - semantic_models)


Semantic-only models: {'highlevel_word2vec_72pcs_conv'}
DF-only models: {'binder_concreteness', 'binder_abstractness', 'word2vec'}


In [33]:
MODEL_MAP = {
    "highlevel_word2vec_72pcs_conv": "word2vec",
    "fasttext_conv": "fasttext",
    # identity mappings are optional but can be explicit:
    "qwen-text_layers28-32_conv": "qwen-text_layers28-32_conv",
    "XLM-roberta_layers11-15": "XLM-roberta_layers11-15",
}
def add_semantic_results(df, semantic):
    df = df.copy()
    df["corr_concreteness"] = np.nan
    df["corr_abstractness"] = np.nan

    for (binder, semantic_model), arr in semantic.items():
        if semantic_model not in MODEL_MAP:
            continue

        df_model = MODEL_MAP[semantic_model]
        col = (
            "corr_concreteness"
            if binder == "binder_concreteness"
            else "corr_abstractness"
        )

        mask = df["model"] == df_model
        if not mask.any():
            continue

        idx = df.loc[mask, "index"].values
        valid = idx < len(arr)

        df.loc[mask[mask].index[valid], col] = arr[idx[valid]]

    return df



In [34]:
final_df = add_semantic_results(df, semanticresults)

In [35]:
final_df

,index,model,value,run,participant,roi,roi_type,participant_group,corr_concreteness,corr_abstractness
0,0,qwen-text_layers28-32_conv,0.027176,1,33,2,language,blind,0.270019,-0.079389
1,1,qwen-text_layers28-32_conv,0.015785,1,33,2,language,blind,0.068488,0.104302
2,2,qwen-text_layers28-32_conv,0.030376,1,33,2,language,blind,-0.055686,-0.057188
3,3,qwen-text_layers28-32_conv,0.023361,1,33,2,language,blind,0.044549,-0.005386
4,4,qwen-text_layers28-32_conv,0.095281,1,33,2,language,blind,0.137672,0.129640
...,...,...,...,...,...,...,...,...,...,...
5945285,1572,word2vec,-0.071355,6,32,visual,visual,ctrlAV,0.019735,0.005785
5945286,1573,word2vec,0.080193,6,32,visual,visual,ctrlAV,0.039018,0.065027
5945287,1574,word2vec,0.001791,6,32,visual,visual,ctrlAV,0.046878,0.060833
5945288,1575,word2vec,0.013398,6,32,visual,visual,ctrlAV,0.036981,0.052690


In [38]:
nan_rows = df[df[["corr_concreteness", "corr_abstractness"]].isna().any(axis=1)]




In [44]:
df[df[["value"]].isna().any(axis=1)]["model"].unique()

array(['qwen-text_layers28-32_conv', 'qwen-text_layers32-36_conv',
       'qwen-text_layers12-16_conv', 'qwen-text_layers8-12_conv',
       'qwen-text_layers20-24_conv', 'qwen-text_layers0-4_conv',
       'qwen-text_layers16-20_conv', 'qwen-text_layers24-28_conv',
       'qwen-text_layers4-8_conv',
       'qwen-omni_audiovideotext_layers12-16_conv',
       'qwen-omni_audiovideotext_layers8-12_conv',
       'qwen-omni_audiovideotext_layers32-36_conv',
       'qwen-omni_audiovideotext_layers28-32_conv',
       'qwen-omni_audiovideotext_layers24-28_conv',
       'qwen-omni_audiovideotext_layers16-20_conv',
       'qwen-omni_audiovideotext_layers0-4_conv',
       'qwen-omni_audiovideotext_layers4-8_conv',
       'qwen-omni_audiovideotext_layers20-24_conv', 'fasttext_conv',
       'binder_abstractness', 'binder_concreteness'], dtype=object)

In [41]:
nan_rows

,index,model,value,run,participant,roi,roi_type,participant_group,corr_concreteness,corr_abstractness
1033,1033,qwen-text_layers28-32_conv,NaN,4,33,2,language,blind,NaN,NaN
1034,1034,qwen-text_layers28-32_conv,NaN,4,33,2,language,blind,NaN,NaN
1035,1035,qwen-text_layers28-32_conv,NaN,4,33,2,language,blind,NaN,NaN
1036,1036,qwen-text_layers28-32_conv,NaN,4,33,2,language,blind,NaN,NaN
1037,1037,qwen-text_layers28-32_conv,NaN,4,33,2,language,blind,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
5936940,1112,fasttext_conv,NaN,4,32,visual,visual,ctrlAV,NaN,NaN
5941602,1043,binder_abstractness,NaN,4,32,visual,visual,ctrlAV,NaN,NaN
5941671,1112,binder_abstractness,NaN,4,32,visual,visual,ctrlAV,NaN,NaN
5943179,1043,binder_concreteness,NaN,4,32,visual,visual,ctrlAV,NaN,NaN


In [43]:
final_df.to_csv("january/results.csv", sep=";", index=False)